# Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/intro/03_control.ipynb)

Official API intro to controllers and design helpers: impedance / proportional /
filtered loops, LQR → `StateFeedbackController`, and a peek at model-based
methods. MPC closed-loop deploy lives in [`intro/06_hybrid.ipynb`](06_hybrid.ipynb)
and [`applications/mpc.ipynb`](../applications/mpc.ipynb).

**Scripts for depth:** `examples/scripts/control/`, `examples/scripts/robotic/`


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


## LQR at an operating point

`minilink.dynamics.catalog` ships ready-to-use plants; `minilink.control` ships
ready-to-wire controllers and design factories. Below: catalog sweep, forced
cart-pole response, then LQR stabilization.


In [ ]:
import numpy as np
from minilink.control.lqr import lqr_at_operating_point
from minilink.dynamics.catalog.aerial.drone import Drone2D
from minilink.dynamics.catalog.pendulum.cartpole import CartPole
from minilink.dynamics.catalog.vehicles.steering import KinematicCar

for plant in (CartPole(), KinematicCar(), Drone2D()):
    print(f"{plant.name:<22} n={plant.n}  m={plant.m}  states={plant.state.labels}")

cartpole = CartPole()
cartpole.compute_forced(lambda t: np.array([3.0 * np.sin(2.0 * t)]))
cartpole.plot_trajectory()


## Control catalog

Controllers are `System` blocks with standard ports (`r`, `y`, `u`).
The quick start used `ImpedanceController`; the catalog also includes
`ImpedanceIntegralController`, `FilteredController`, and LQR design helpers such as
`lqr_at_operating_point` — linearize at an equilibrium and return a
`StateFeedbackController` ready to wire with `@`.


In [ ]:
import numpy as np
from minilink.dynamics.catalog.pendulum.cartpole import CartPole
from minilink.control.lqr import lqr_at_operating_point

plant = CartPole()
x_bar = np.array([0.0, np.pi, 0.0, 0.0])
Q = np.diag([1.0, 1.0, 1.0, 1.0])
R = np.array([[1.0]])

lqr_ctl = lqr_at_operating_point(plant, x_bar, Q, R)
lqr_loop = lqr_ctl @ plant

plant.x0 = np.array([-3.0, np.pi - 0.3, 0.0, 0.0])
lqr_loop.compute_trajectory(tf=8.0)
lqr_loop.plot_trajectory()


## Model-based peek

Computed-torque and sliding-mode demos live under `examples/scripts/control/`.
Robotic impedance / kinematic / nullspace stacks: `examples/scripts/robotic/`.


In [ ]:
from minilink.control.modelbased import ComputedTorqueController, SlidingModeController

print(ComputedTorqueController.__name__, SlidingModeController.__name__)
print("Demos: examples/scripts/control/demo_computed_torque_pendulum.py")
print("       examples/scripts/control/demo_sliding_mode_pendulum.py")
print("Robotic: examples/scripts/robotic/")
